In [1]:
!git clone https://github.com/aitf-its-tim3-dfk/david.git

fatal: destination path 'david' already exists and is not an empty directory.


In [1]:
%cd david

/content/david


In [2]:
!git switch feat/refactor-farhan

M	config.yaml
M	dataset.py
M	encoder.py
Already on 'feat/refactor-farhan'
Your branch is up to date with 'origin/feat/refactor-farhan'.


In [4]:
%pip install ftfy regex decord scikit-learn yacs

In [3]:
!pip install -U kaggle

In [20]:
import kagglehub
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
%env KAGGLE_USERNAME=$username
%env KAGGLE_KEY=$key

env: KAGGLE_USERNAME=farhanwew
env: KAGGLE_KEY=51376389df8a5a095fd393132ef1b88f


In [ ]:
!sudo apt install aria2 -y

In [ ]:
!aria2c -x 16 -s 16 -k 1M \
--http-user=$KAGGLE_USERNAME \
--http-passwd=$KAGGLE_KEY \
--continue=true \
--max-connection-per-server=16 \
--split=16 \
--min-split-size=10M \
-o real-vs-gen-videos.zip \
"https://www.kaggle.com/api/v1/datasets/download/farhanwew/real-vs-gen-videos"

In [ ]:
# Install
!sudo apt install pigz

In [ ]:
%%capture
!unzip real-vs-gen-videos.zip -d final_dataset

In [ ]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1Nx30Kbu5xnv6dPwz4I3Ivy380LCdp1Md

In [3]:
%%writefile config.yaml

# DAViD Configuration
# All fields are optional — missing keys fall back to defaults in config.py.
#
# ── Legacy Google Drive setup (pre-refactor) ──────────────────────────────────
# To revert to the old structure (Drive mount + JSON cache instead of CSV),
# replace the paths section with the following and call get_train_val_loaders()
# without csv_path:
#
#   paths:
#     base_dir: /content/drive/MyDrive/data mining gemastik
#     best_model_save_path: best_detector_model.pt
#     vificlip_checkpoint: /content/drive/MyDrive/vifi_clip_30_epochs_k400_full_finetuned.pth
#
# Then copy the cache locally in the notebook before training:
#   CACHE_PATH       = "video_train_10000_cache_fixed_2.json"
#   DRIVE_CACHE_PATH = "/content/drive/MyDrive/data mining gemastik/video train 10000/video_train_10000_cache_fixed_2.json"
#   if not os.path.exists(CACHE_PATH):
#       shutil.copy(DRIVE_CACHE_PATH, CACHE_PATH)
#
# Real video directories (legacy):
#   /content/drive/.shortcut-targets-by-id/1Wcbv564DV62urzCJvYmvnkDD_Z74ZKLa/GenVideo-Val/Real
#   /content/drive/MyDrive/data mining gemastik/K4/videos_val
#
# Fake video directories (legacy):
#   /content/drive/MyDrive/gemastik-datmin/pika/train_pika
#   /content/drive/MyDrive/data mining gemastik/Sora/train_OpenSora
#   /content/drive/MyDrive/data mining gemastik/SVD/train_SVD
# ─────────────────────────────────────────────────────────────────────────────

paths:
  base_dir: final_dataset
  metadata_csv: final_dataset/metadata.csv
  best_model_save_path: best_detector_model.pt
  vificlip_checkpoint: k400_clip_complete_finetuned_30_epochs.pth

model:
  clip_arch: ViT-B/16
  class_names: ["true", "false"]
  head_type: deep          # "simple" (1-layer) | "deep" (3-layer)

training:
  learning_rate: 1.0e-3
  batch_size: 64
  num_epochs: 1
  num_frames: 16
  num_workers: 4
  input_dim: 512
  num_classes: 1
  use_amp: true            # mixed precision — disable if running on CPU
  lr_scheduler: cosine     # "cosine" | "plateau" | null
  patience: 3              # early stopping epochs; null to disable

dataset:
  val_split: 0.2           # used when train_size and val_size are both null
  train_size: 20000         # e.g. 15000 — null means use all remaining after val
  val_size: 1000        # e.g. 200   — null means use val_split ratio


Overwriting config.yaml


In [4]:
from config import CLIP_ARCH, CLASS_NAMES, VIFICLIP_CHECKPOINT
from encoder import load_feature_extractor

feature_extractor = load_feature_extractor(
    arch=CLIP_ARCH,
    class_names=CLASS_NAMES,
    checkpoint_path=VIFICLIP_CHECKPOINT,
).cuda()

In [5]:
import os
from config import METADATA_CSV, BASE_DIR, BATCH_SIZE, VAL_SPLIT, NUM_WORKERS
from config import TRAIN_SIZE, VAL_SIZE
from config import INPUT_DIM, NUM_CLASSES, LEARNING_RATE, NUM_EPOCHS, BEST_MODEL_SAVE_PATH, HEAD_TYPE

In [7]:
from model import build_transform
from dataset import get_train_val_loaders
from train import set_seed

set_seed(42)

CLEAN_CSV = "final_dataset/metadata_clean_v2.csv"

preprocess = build_transform(224)
train_loader, val_loader, val_files = get_train_val_loaders(
    transform=preprocess,
    csv_path=METADATA_CSV,
    base_dir=BASE_DIR,
    val_split=VAL_SPLIT,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    validate = True,
    clean_csv_path=CLEAN_CSV,  # save clean CSV once, reuse next time
)

Validating videos: 100%|██████████| 81925/81925 [24:37<00:00, 55.45it/s]


Loaded 79630 videos from final_dataset/metadata.csv (2295 broken files skipped)
Clean CSV saved: final_dataset/metadata_clean_v2.csv
Dataset split: 20000 training files, 1000 validation files.


In [8]:
from config import USE_AMP, LR_SCHEDULER, PATIENCE
from train import set_seed, run_training

set_seed(42)

classifier = run_training(
    feature_extractor=feature_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    head_type=HEAD_TYPE,
    lr=LEARNING_RATE,
    num_epochs=5,
    save_path=BEST_MODEL_SAVE_PATH,
    use_amp=USE_AMP,
    lr_scheduler=LR_SCHEDULER,
    patience=PATIENCE,
)

Using device: cuda | AMP: True | Scheduler: cosine | Patience: 3
Creating classification head: 'deep'

--- Epoch 1/5 | Training ---


100%|██████████| 313/313 [29:48<00:00,  5.72s/it]


Average Training Loss: 0.2747
--- Epoch 1/5 | Validation ---


100%|██████████| 16/16 [01:45<00:00,  6.57s/it]

Average Validation Loss: 0.1048
Validation Accuracy: 95.80%
LR: 9.05e-04
New best model saved with accuracy: 95.80%

--- Epoch 2/5 | Training ---


...

Average Training Loss: 0.0493
--- Epoch 5/5 | Validation ---


100%|██████████| 16/16 [01:44<00:00,  6.55s/it]


Average Validation Loss: 0.0794
Validation Accuracy: 97.40%
LR: 0.00e+00

--- Training Complete ---
Loading best model from best_detector_model.pt for final report...


100%|██████████| 16/16 [01:45<00:00,  6.57s/it]

Average Validation Loss: 0.0701
Validation Accuracy: 97.40%

--- Final Performance Report (on validation set) ---
              precision    recall  f1-score   support

    Fake (0)       0.99      0.97      0.98       653
    Real (1)       0.95      0.98      0.96       347

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



In [9]:
from evaluate import evaluate
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics = evaluate(classifier, feature_extractor, val_loader, val_files, device)

EVALUATION REPORT


Evaluating: 100%|██████████| 16/16 [01:44<00:00,  6.54s/it]


AUC-ROC : 0.9958
Optimal threshold : 0.505  (F1=0.9654  vs  F1@0.5=0.9654)

── Classification Report (threshold=0.5) ──
              precision    recall  f1-score   support

    Fake (0)       0.98      0.98      0.98       653
    Real (1)       0.97      0.97      0.97       347

    accuracy                           0.98      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.98      0.98      0.98      1000

── Confusion Matrix (threshold=0.5) ──
              Pred Fake  Pred Real
  True Fake         641         12
  True Real          12        335

  False Positive Rate : 0.018  (real misclassified as fake)
  False Negative Rate : 0.035  (fake slipping through)

── Per-source Accuracy ──
  K4              n=  252   acc=96.4%   auc=nan
  MSSRVT          n=   92   acc=96.7%   auc=nan
  SD              n=  116   acc=99.1%   auc=nan
  SVD             n=  119   acc=99.2%   auc=nan
  Sora            n=  119   acc=98.3%   auc=nan
  ZeroScope       n=  111


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dis

In [10]:
!mkdir result

In [11]:
%cd result

/content/david/result


In [13]:
%%writefile summary.txt
============================================================
EVALUATION REPORT
============================================================
Evaluating: 100%|██████████| 16/16 [01:44<00:00,  6.54s/it]
AUC-ROC : 0.9958
Optimal threshold : 0.505  (F1=0.9654  vs  F1@0.5=0.9654)

── Classification Report (threshold=0.5) ──
              precision    recall  f1-score   support

    Fake (0)       0.98      0.98      0.98       653
    Real (1)       0.97      0.97      0.97       347

    accuracy                           0.98      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.98      0.98      0.98      1000

── Confusion Matrix (threshold=0.5) ──
              Pred Fake  Pred Real
  True Fake         641         12
  True Real          12        335

  False Positive Rate : 0.018  (real misclassified as fake)
  False Negative Rate : 0.035  (fake slipping through)

── Per-source Accuracy ──
  K4              n=  252   acc=96.4%   auc=nan
  MSSRVT          n=   92   acc=96.7%   auc=nan
  SD              n=  116   acc=99.1%   auc=nan
  SVD             n=  119   acc=99.2%   auc=nan
  Sora            n=  119   acc=98.3%   auc=nan
  ZeroScope       n=  111   acc=98.2%   auc=nan
  coin            n=    3   acc=100.0%   auc=nan
  pika            n=  131   acc=96.9%   auc=nan
  tiktok          n=   57   acc=96.5%   auc=nan
============================================================

Writing summary.txt


In [15]:
!cp /content/david/best_detector_model.pt .

In [19]:
%%writefile dataset-metadata.json
{
  "title": "Training Artficat",
  "id": "farhanwew/Training-Artficat-david",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}

Overwriting dataset-metadata.json


In [17]:
!ls

best_detector_model.pt	dataset-metadata.json  summary.txt


In [21]:
!kaggle datasets create -p "." --dir-mode tar

Starting upload for file summary.txt
100% 1.44k/1.44k [00:00<00:00, 2.31kB/s]
Upload successful: summary.txt (1KB)
Starting upload for file best_detector_model.pt
100% 3.01M/3.01M [00:00<00:00, 4.57MB/s]
Upload successful: best_detector_model.pt (3MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/farhanwew/Training-Artficat-david
